In [1]:
from pathlib import Path

import openpyxl
import pandas as pd

FILE = Path('LPCON - Nurban 901 - REV.01.xlsx')

# Carrega com data_only=True para resolver fórmulas (usa valor cacheado, não a fórmula)
wb = openpyxl.load_workbook(FILE, data_only=True)
print("Sheets:", wb.sheetnames)

# Lê cada sheet em um dict de DataFrames (header=None para ver tudo bruto)
sheets_raw = pd.read_excel(FILE, sheet_name=None, header=None, engine='openpyxl')
print("Shapes brutas:", {k: v.shape for k, v in sheets_raw.items()})

Sheets: ['familias', 'aditivos', 'lançamentos', 'orçamento', 'Aux', 'resumo']
Shapes brutas: {'familias': (10, 2), 'aditivos': (3, 11), 'lançamentos': (202, 12), 'orçamento': (61, 19), 'Aux': (24, 10), 'resumo': (15, 7)}


/opt/homebrew/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
/opt/homebrew/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [2]:
## Comandos essenciais para manipular este tipo de arquivo

# 1. Ver primeiras linhas brutas de uma sheet (sem header automático)
df_bruto = sheets_raw['lançamentos']
print("=== lançamentos — primeiras 6 linhas brutas ===")
print(df_bruto.head(6).to_string())
print()

# 2. Contar células não-nulas por linha (para detectar onde está o header real)
nao_nulos_por_linha = df_bruto.notna().sum(axis=1)
print("Células não-nulas por linha (primeiras 8):")
print(nao_nulos_por_linha.head(8).to_dict())

=== lançamentos — primeiras 6 linhas brutas ===
            0        1                    2                      3           4                          5                              6            7        8   9      10  11
0          NaN      NaN                  NaN                    NaN         NaN                        NaN                            NaN          NaN      NaN NaN    NaN NaN
1     MATERIAL  INSUMOS                  NaN                    NaN         NaN                        NaN                            NaN          NaN      NaN NaN  VENDA NaN
2  MÃO DE OBRA        #           DATA CUSTO             FORNECEDOR  MAT ou MDO  CENTRO DE CUSTO - FAMILIA  CENTRO DE CUSTO - SUB FAMILIA  NÚMERO   NF    VALOR NaN    NaN NaN
3          NaN        1  2025-09-12 00:00:00        BAZAR TODA OBRA         MAT                 OBRA CIVIL          ELEVAÇÃO DE ALVENARIA          NaN     21.9 NaN    NaN NaN
4          NaN        2  2025-09-12 00:00:00            A ILUMINADA         M

In [3]:
## 3. Detectar header row automaticamente: linha com mais células não-nulas nas primeiras 10 rows

def detect_header_row(df_raw, max_search=10):
    sample = df_raw.iloc[:max_search]
    counts = sample.notna().sum(axis=1)
    best_idx = counts.idxmax()
    print(f"  Não-nulos por linha: {counts.to_dict()}")
    print(f"  → Header na posição {best_idx} (row {best_idx + 1} do Excel)")
    return best_idx

for sheet_name in ['lançamentos', 'aditivos', 'orçamento', 'resumo']:
    print(f"\nSheet '{sheet_name}':")
    detect_header_row(sheets_raw[sheet_name])


Sheet 'lançamentos':
  Não-nulos por linha: {0: 0, 1: 3, 2: 9, 3: 7, 4: 7, 5: 7, 6: 7, 7: 7, 8: 8, 9: 8}
  → Header na posição 2 (row 3 do Excel)

Sheet 'aditivos':
  Não-nulos por linha: {0: 0, 1: 2, 2: 10}
  → Header na posição 2 (row 3 do Excel)

Sheet 'orçamento':
  Não-nulos por linha: {0: 0, 1: 1, 2: 17, 3: 6, 4: 7, 5: 10, 6: 10, 7: 10, 8: 7, 9: 10}
  → Header na posição 2 (row 3 do Excel)

Sheet 'resumo':
  Não-nulos por linha: {0: 0, 1: 5, 2: 3, 3: 4, 4: 4, 5: 4, 6: 4, 7: 4, 8: 4, 9: 4}
  → Header na posição 1 (row 2 do Excel)


In [4]:
## 4. Extrair lançamentos limpos usando a detecção automática

def parse_sheet(df_raw, sheet_name, max_search=10):
    header_idx = detect_header_row(df_raw, max_search)
    headers = df_raw.iloc[header_idx].tolist()
    # Nomear colunas sem header como col_N
    headers = [str(h).strip() if pd.notna(h) and str(h).strip() not in ('', 'None') else f'col_{i}'
               for i, h in enumerate(headers)]
    df = df_raw.iloc[header_idx + 1:].copy()
    df.columns = headers
    df = df.dropna(how='all')
    # Descartar colunas totalmente vazias ou sem nome real
    df = df.loc[:, ~df.columns.str.startswith('col_')]
    df = df.reset_index(drop=True)
    return df, headers

print("=== lançamentos ===")
df_lanc, h = parse_sheet(sheets_raw['lançamentos'], 'lançamentos')
print(f"Shape: {df_lanc.shape}")
print(f"Colunas: {list(df_lanc.columns)}")
df_lanc.head()

=== lançamentos ===
  Não-nulos por linha: {0: 0, 1: 3, 2: 9, 3: 7, 4: 7, 5: 7, 6: 7, 7: 7, 8: 8, 9: 8}
  → Header na posição 2 (row 3 do Excel)
Shape: (199, 9)
Colunas: ['MÃO DE OBRA', '#', 'DATA CUSTO', 'FORNECEDOR', 'MAT ou MDO', 'CENTRO DE CUSTO - FAMILIA', 'CENTRO DE CUSTO - SUB FAMILIA', 'NÚMERO   NF', 'VALOR']


,MÃO DE OBRA,#,DATA CUSTO,FORNECEDOR,MAT ou MDO,CENTRO DE CUSTO - FAMILIA,CENTRO DE CUSTO - SUB FAMILIA,NÚMERO NF,VALOR
0,NaN,1,2025-09-12 00:00:00,BAZAR TODA OBRA,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,NaN,21.9
1,NaN,2,2025-09-12 00:00:00,A ILUMINADA,MAT,INSTALAÇÕES,INSTALAÇÃO ELÉTRICA,NaN,3216.39
2,NaN,3,2025-09-12 00:00:00,EDSON SANTOS MATERIAL,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,NaN,30
3,NaN,4,2025-09-12 00:00:00,EDSON SANTOS MATERIAL,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,NaN,38.75
4,NaN,5,2025-09-12 00:00:00,ÁGUAS CLARA,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS,NaN,36


In [5]:
## 5. Problema de fórmulas: coluna # tem =B4+1 em vez do número calculado
# data_only=True no openpyxl usa o cache — mas pd.read_excel também resolve?

print("Coluna '#' — primeiros valores:")
print(df_lanc['#'].head(10).tolist())
print()

# Verificar se DATA CUSTO está como datetime ou string
print("Tipo de DATA CUSTO:", df_lanc['DATA CUSTO'].dtype)
print(df_lanc['DATA CUSTO'].head(5).tolist())

Coluna '#' — primeiros valores:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Tipo de DATA CUSTO: object
[datetime.datetime(2025, 9, 12, 0, 0), datetime.datetime(2025, 9, 12, 0, 0), datetime.datetime(2025, 9, 12, 0, 0), datetime.datetime(2025, 9, 12, 0, 0), datetime.datetime(2025, 9, 12, 0, 0)]


In [6]:
## 6. Limpeza e tipagem final para o schema blu

df_clean = pd.DataFrame()
df_clean['data']         = pd.to_datetime(df_lanc['DATA CUSTO'], errors='coerce').dt.date
df_clean['fornecedor']   = df_lanc['FORNECEDOR'].astype(str).str.strip().replace('nan', pd.NA)
df_clean['numero_nf']    = df_lanc['NÚMERO   NF'].astype(str).str.strip().replace({'nan': pd.NA, 'None': pd.NA})
df_clean['valor']        = pd.to_numeric(df_lanc['VALOR'], errors='coerce')
df_clean['tipo']         = df_lanc['MAT ou MDO'].str.strip().str.upper()
df_clean['categoria']    = df_lanc['CENTRO DE CUSTO - FAMILIA'].str.strip()
df_clean['subcategoria'] = df_lanc['CENTRO DE CUSTO - SUB FAMILIA'].str.strip()

# Remover linhas sem data ou valor (fórmulas vazias, subtotais, etc.)
df_clean = df_clean.dropna(subset=['data', 'valor'])
df_clean = df_clean[df_clean['valor'] > 0]
df_clean = df_clean.reset_index(drop=True)

print(f'Registros válidos: {len(df_clean)}')
print(f'Período: {df_clean["data"].min()} → {df_clean["data"].max()}')
print(f'Total MAT: R$ {df_clean[df_clean.tipo=="MAT"]["valor"].sum():,.2f}')
print(f'Total MDO: R$ {df_clean[df_clean.tipo=="MDO"]["valor"].sum():,.2f}')
print(f'Total geral: R$ {df_clean["valor"].sum():,.2f}')
df_clean.head(10)


Registros válidos: 199
Período: 2025-09-12 → 2026-04-24
Total MAT: R$ 54,733.41
Total MDO: R$ 46,261.90
Total geral: R$ 100,995.31


,data,fornecedor,numero_nf,valor,tipo,categoria,subcategoria
0,2025-09-12,BAZAR TODA OBRA,NaN,21.90,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA
1,2025-09-12,A ILUMINADA,NaN,3216.39,MAT,INSTALAÇÕES,INSTALAÇÃO ELÉTRICA
2,2025-09-12,EDSON SANTOS MATERIAL,NaN,30.00,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA
3,2025-09-12,EDSON SANTOS MATERIAL,NaN,38.75,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA
4,2025-09-12,ÁGUAS CLARA,NaN,36.00,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS
5,2025-09-12,ZAMACON,33745,240.00,MAT,DEMOLIÇÕES,DEMOLIÇÃO DE ALVENARIA
6,2025-09-12,ZAMACON,33745,1058.50,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA
7,2025-09-12,AL GOSTO,NaN,60.00,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS
8,2025-09-12,ZAMACON,33745,916.80,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,"EQUIPAMENTOS, EPIS E EPCS"
9,2025-09-15,ZAMACON,33796,1049.00,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA


In [7]:
## 7. Qualidade dos dados e valores únicos

print('=== Nulos por coluna ===')
print(df_clean.isnull().sum())
print()
print('=== Tipos únicos (MAT/MDO) ===')
print(df_clean['tipo'].value_counts(dropna=False))
print()
print('=== Categorias únicas ===')
print(df_clean['categoria'].value_counts())


=== Nulos por coluna ===
data              0
fornecedor        0
numero_nf       168
valor             0
tipo              0
categoria         0
subcategoria      7
dtype: int64

=== Tipos únicos (MAT/MDO) ===
tipo
MAT    161
MDO     38
Name: count, dtype: int64

=== Categorias únicas ===
categoria
MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO    69
OBRA CIVIL                              61
INSTALAÇÕES                             37
DEMOLIÇÕES                              12
ESTRUTURA E DE CONCRETO ARMADO           9
REVESTIMENTOS DE PISO E PAREDES          6
PINTURA                                  5
Name: count, dtype: int64


In [8]:
## 8. Sheet aditivos — mesma lógica

print('=== aditivos ===')
df_adit, _ = parse_sheet(sheets_raw['aditivos'], 'aditivos')
print(f'Shape: {df_adit.shape}')
print(f'Colunas: {list(df_adit.columns)}')

# Filtrar só linhas com VALOR
df_adit = df_adit[pd.to_numeric(df_adit['VALOR'], errors='coerce').notna()]
df_adit = df_adit[pd.to_numeric(df_adit['VALOR'], errors='coerce') > 0]
print(f'Registros com valor: {len(df_adit)}')
print(f'Total R$: R$ {pd.to_numeric(df_adit["VALOR"], errors="coerce").sum():,.2f}')
df_adit.head()


=== aditivos ===
  Não-nulos por linha: {0: 0, 1: 2, 2: 10}
  → Header na posição 2 (row 3 do Excel)
Shape: (0, 10)
Colunas: ['DATA DE RECEBIMENTO', 'QUINZENA', 'FORNECEDOR', 'QTD', 'UNID.', 'DESCRIÇÃO', 'MARCA', 'CENTRO DE CUSTO', 'NÚMERO   NF', 'VALOR']
Registros com valor: 0
Total R$: R$ 0.00


,DATA DE RECEBIMENTO,QUINZENA,FORNECEDOR,QTD,UNID.,DESCRIÇÃO,MARCA,CENTRO DE CUSTO,NÚMERO NF,VALOR


In [9]:
## 9. Consolidar: lançamentos + aditivos em um único DataFrame pronto para ingestão

# aditivos: renomear colunas para o mesmo schema
df_adit_clean = pd.DataFrame()
df_adit_clean['data']         = pd.to_datetime(df_adit['DATA DE RECEBIMENTO'], errors='coerce').dt.date
df_adit_clean['fornecedor']   = df_adit['FORNECEDOR'].astype(str).str.strip().replace('nan', pd.NA)
df_adit_clean['numero_nf']    = df_adit['NÚMERO   NF'].astype(str).str.strip().replace({'nan': pd.NA, 'None': pd.NA})
df_adit_clean['valor']        = pd.to_numeric(df_adit['VALOR'], errors='coerce')
df_adit_clean['tipo']         = 'ADITIVO'
df_adit_clean['categoria']    = df_adit['CENTRO DE CUSTO']
df_adit_clean['subcategoria'] = df_adit['DESCRIÇÃO'].astype(str).str.strip()
df_adit_clean = df_adit_clean.dropna(subset=['data', 'valor'])
df_adit_clean = df_adit_clean[df_adit_clean['valor'] > 0]

df_all = pd.concat([df_clean, df_adit_clean], ignore_index=True)
print(f'Total consolidado: {len(df_all)} registros')
print(f'Total R$: R$ {df_all["valor"].sum():,.2f}')
print()
print(df_all['tipo'].value_counts())
df_all.tail()


Total consolidado: 199 registros
Total R$: R$ 100,995.31

tipo
MAT    161
MDO     38
Name: count, dtype: int64


,data,fornecedor,numero_nf,valor,tipo,categoria,subcategoria
194,2026-04-14,ZAMACON,NaN,380.40,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA
195,2026-04-14,UBER FLASH,NaN,40.95,MAT,PINTURA,PINTURA ACRÍLICA BRANCA
196,2026-04-20,ZAMACON,NaN,2017.30,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA
197,2026-04-24,ANDRÉ AJUDANTE,NaN,160.00,MDO,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA
198,2026-04-24,LUIS ENCARREGADO,NaN,340.00,MDO,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA


In [10]:
## 10. Resumo: o que precisamos mudar no pipeline blu

# Este arquivo tem:
#  - Múltiplas sheets relevantes (lançamentos + aditivos)
#  - 2 linhas de 'header falso' antes do header real (labels de seção + row vazia)
#  - Fórmulas na coluna # (resolvidas via data_only=True no openpyxl)
#  - Campos sem equivalente direto no schema atual (tipo MAT/MDO, subcategoria)

# Mudanças necessárias no pipeline upload-csv-source:
#  1. Para XLSX com múltiplas sheets → perguntar qual sheet o usuário quer usar
#  2. detect_header_row: usar 'row com max células não-nulas' (já implementado)
#  3. Filtrar linhas de dados onde VALOR é nulo ou <= 0
#  4. Filtrar colunas sem nome real (col_N)

# Campos gerados e seu mapeamento para colunas blu:
mapeamento = {
    'DATA CUSTO':                    'data',
    'FORNECEDOR':                    'fornecedor',
    'NÚMERO   NF':                   'numero_nf',
    'VALOR':                         'valor',
    'MAT ou MDO':                    'tipo_lancamento',   # novo campo no schema
    'CENTRO DE CUSTO - FAMILIA':     'categoria',         # novo campo
    'CENTRO DE CUSTO - SUB FAMILIA': 'subcategoria',      # novo campo
}

print('Mapeamento proposto:')
for src, dst in mapeamento.items():
    print(f'  {src:45s} → {dst}')

print()
print('Schema gaps (campos novos que precisam ser adicionados ao schema blu):')
novos = ['tipo_lancamento (MAT/MDO/ADITIVO)', 'categoria (centro de custo família)', 'subcategoria (centro de custo sub-família)']
for n in novos:
    print(f'  ⚠  {n}')


Mapeamento proposto:
  DATA CUSTO                                    → data
  FORNECEDOR                                    → fornecedor
  NÚMERO   NF                                   → numero_nf
  VALOR                                         → valor
  MAT ou MDO                                    → tipo_lancamento
  CENTRO DE CUSTO - FAMILIA                     → categoria
  CENTRO DE CUSTO - SUB FAMILIA                 → subcategoria

Schema gaps (campos novos que precisam ser adicionados ao schema blu):
  ⚠  tipo_lancamento (MAT/MDO/ADITIVO)
  ⚠  categoria (centro de custo família)
  ⚠  subcategoria (centro de custo sub-família)
